In [1]:
import math
import time
import numpy as np
import pandas as pd
import yfinance as yf
import os
import sys


import torch
import torch.nn as nn


REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, REPO_ROOT)
from model import train_pinn


from montecarlo import mc_asian_call_arith
from montecarlo import pinn_price_real


from evaluation import evaluate_mae_pinn_vs_mc
from evaluation import parity_plot_plotly
from evaluation import run_full_evaluation


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32


In [2]:
# Get real WTI data and infer ranges

df = yf.download("CL=F", start="2010-01-01", progress=False)
prices = df["Close"].dropna()


S0 = float(prices.iloc[-1])
S_max_hist = float(prices.max())


log_ret = np.log(prices / prices.shift(1)).dropna()
sigma_hist = float(log_ret.std()) * np.sqrt(252)


T = 1.0
S_max_real = 1.2 * S_max_hist
I_max_real = S_max_real * T
sigma_max = min(1.0, 2.0 * sigma_hist)
r_max = 0.5


print("=== REAL (USD) ranges ===")
print(f"S0        = {S0:.2f}")
print(f"S_max     = {S_max_real:.2f}")
print(f"I_max     = {I_max_real:.2f}")
print(f"sigma     = {sigma_hist:.3f}")
print(f"sigma_max = {sigma_max:.3f}")
print(f"r_max     = {r_max}")


YF.download() has changed argument auto_adjust default to True
=== REAL (USD) ranges ===
S0        = 57.32
S_max     = 148.44
I_max     = 148.44
sigma     = 0.404
sigma_max = 0.807
r_max     = 0.5


/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_85158/968340630.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(prices.iloc[-1])
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_85158/968340630.py:8: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S_max_hist = float(prices.max())
/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_85158/968340630.py:12: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  sigma_hist = float(log_ret.std()) * np.sqrt(252)


In [3]:
# Normalization: K_scale = S0  (so S̃0 = 1)

K_scale = S0  


S_max = S_max_real / K_scale
I_max = I_max_real / K_scale


print("\n=== NORMALIZED (dimensionless) ranges ===")
print(f"K_scale   = {K_scale:.2f}")
print(f"S0_tilde  = {S0 / K_scale:.3f}")      # should be 1.000
print(f"S_max_t   = {S_max:.3f}")
print(f"I_max_t   = {I_max:.3f}")
print(f"r_max     = {r_max:.3f}")
print(f"sigma_max = {sigma_max:.3f}")



=== NORMALIZED (dimensionless) ranges ===
K_scale   = 57.32
S0_tilde  = 1.000
S_max_t   = 2.590
I_max_t   = 2.590
r_max     = 0.500
sigma_max = 0.807


In [8]:
model = train_pinn(
   S_max, I_max, r_max, sigma_max,
   width=160,
   depth=4,
   n_epochs=50000,     # start here; increase toward 200k/500k if needed
   lr0=1e-3,
   Np=1000,
   n_bc_axis=100,
   w_pde=0.6,
   print_every=2000
)

K_real = S0  # ATM
r_eval = 0.05
sigma_eval = sigma_hist



print("Evaluation (real scale)")
test_S = [0.8*S0, 1.0*S0, 1.2*S0]  # around spot


for S_test in test_S:
   pinn_p = pinn_price_real(model, S_test, K_real, r_eval, sigma_eval, t0=0.0)
   mc_p, mc_se = mc_asian_call_arith(S_test, K_real, r_eval, sigma_eval, T=1.0, n_steps=252, n_paths=50_000, antithetic=True)


   print(f"S={S_test:8.2f} | PINN={pinn_p:10.4f} | MC={mc_p:10.4f} ± {1.645*mc_se:8.4f} (90% CI half-width)")


ep=   2000 | loss=5.438e-04 | pde=4.042e-04 | data=3.013e-04 | lr=9.96e-04 | 2.8 min
ep=   4000 | loss=2.060e-04 | pde=1.762e-04 | data=1.003e-04 | lr=9.84e-04 | 5.3 min
ep=   6000 | loss=9.268e-05 | pde=7.717e-05 | data=4.637e-05 | lr=9.65e-04 | 7.8 min
ep=   8000 | loss=5.826e-05 | pde=3.826e-05 | data=3.530e-05 | lr=9.38e-04 | 10.3 min
ep=  10000 | loss=3.924e-05 | pde=2.859e-05 | data=2.209e-05 | lr=9.05e-04 | 12.9 min
ep=  12000 | loss=3.123e-05 | pde=1.964e-05 | data=1.944e-05 | lr=8.64e-04 | 15.3 min
ep=  14000 | loss=7.906e-05 | pde=1.372e-05 | data=7.082e-05 | lr=8.19e-04 | 17.9 min
ep=  16000 | loss=6.939e-05 | pde=1.385e-05 | data=6.108e-05 | lr=7.68e-04 | 20.4 min
ep=  18000 | loss=1.592e-05 | pde=1.121e-05 | data=9.190e-06 | lr=7.13e-04 | 22.9 min
ep=  20000 | loss=1.370e-05 | pde=9.019e-06 | data=8.288e-06 | lr=6.55e-04 | 25.4 min
ep=  22000 | loss=1.347e-05 | pde=1.019e-05 | data=7.358e-06 | lr=5.94e-04 | 28.0 min
ep=  24000 | loss=9.113e-06 | pde=9.351e-06 | data=3.503e

In [9]:
S_grid_eval = np.linspace(0, S_max_real, 100)

results = run_full_evaluation(
    model=model,          # PINN già addestrata
    S_grid=S_grid_eval,   # grid su S
    K=K_real,        # strike reale
    r=r_eval,             # risk-free rate
    sigma=sigma_eval,     # volatilità
    T=1.0,                # maturità
    n_steps=252,       # discretizzazione MC
    n_paths=200_000,   # paths MC
    h_delta=1.0,          # bump per delta
)

MAE (PINN vs MC): 0.259498
Speed-up (MC / PINN): 19044.4x


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# ============================================================
# 1) Download WTI data (continuous futures)
# ============================================================
ticker = "CL=F"
start  = "2015-01-01"
end    = None   # up to today

df = yf.download(ticker, start=start, end=end, progress=False)
df = df[["Close"]].dropna()
df.rename(columns={"Close": "price"}, inplace=True)

# ============================================================
# 2) Log-returns
# ============================================================
df["log_ret"] = np.log(df["price"] / df["price"].shift(1))
df = df.dropna()

# Time step (daily)
DT = 1.0 / 252.0

# ============================================================
# 3) Robust diffusion volatility estimate
#    (winsorized to reduce jump contamination)
# ============================================================
def winsorized_std(x, lower=0.01, upper=0.99):
    lo, hi = np.quantile(x, [lower, upper])
    x_w = np.clip(x, lo, hi)
    return x_w.std(ddof=1)

sigma_diff = winsorized_std(df["log_ret"].values) / np.sqrt(DT)

# ============================================================
# 4) Jump detection via threshold rule
# ============================================================
C = 4.0  # threshold multiplier (3–5 is standard)
threshold = C * sigma_diff * np.sqrt(DT)

df["is_jump"] = np.abs(df["log_ret"]) > threshold

# ============================================================
# 5) Estimate lambda (jump intensity)
# ============================================================
n_jumps = df["is_jump"].sum()
n_days  = len(df)
T_years = n_days / 252.0

lambda_jump = n_jumps / T_years

# ============================================================
# 6) Estimate jump size distribution
#    Y ≈ log-return on jump days
# ============================================================
Y = df.loc[df["is_jump"], "log_ret"].values

mu_J    = Y.mean()
sigma_J = Y.std(ddof=1)

# ============================================================
# 7) Jump compensator
# ============================================================
kappa = np.exp(mu_J + 0.5 * sigma_J**2) - 1.0

# ============================================================
# 8) Results
# ============================================================
print("=== Jump-Diffusion Parameter Estimates (WTI) ===")
print(f"Sample period: {df.index.min().date()} → {df.index.max().date()}")
print(f"Observations: {n_days}")
print(f"Detected jumps: {n_jumps}")
print("")
print(f"lambda_jump  = {lambda_jump:.3f}  (jumps / year)")
print(f"mu_J         = {mu_J:.4f}")
print(f"sigma_J      = {sigma_J:.4f}")
print(f"kappa        = {kappa:.4f}")


YF.download() has changed argument auto_adjust default to True
=== Jump-Diffusion Parameter Estimates (WTI) ===
Sample period: 2015-01-05 → 2026-01-02
Observations: 2764
Detected jumps: 24

lambda_jump  = 2.188  (jumps / year)
mu_J         = 0.0196
sigma_J      = 0.1817
kappa        = 0.0368


/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [10]:
import torch
import numpy as np
import math

# ============================================================
# Setup
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

# ============================================================
# Monte Carlo Asian Call – Merton Jump–Diffusion
# ============================================================

def mc_asian_call_arith_jump(
    S0_real, K_real, r, sigma,
    lambda_jump, mu_J, sigma_J,
    T=1.0,
    n_steps=252,
    n_paths=200_000,
    antithetic=True
):
    dt = T / n_steps
    disc = math.exp(-r * T)

    # Brownian part
    if antithetic:
        half = n_paths // 2
        Z = np.random.normal(size=(half, n_steps))
        Z = np.vstack([Z, -Z])
    else:
        Z = np.random.normal(size=(n_paths, n_steps))

    n_sim = Z.shape[0]

    # Poisson jumps
    N_jump = np.random.poisson(lambda_jump * dt, size=(n_sim, n_steps))

    S = np.full((n_sim,), S0_real, dtype=float)
    sumS = np.zeros_like(S)

    # Drift compensation
    kappa = math.exp(mu_J + 0.5 * sigma_J**2) - 1.0
    drift = (r - lambda_jump * kappa - 0.5 * sigma**2) * dt
    vol = sigma * math.sqrt(dt)

    for k in range(n_steps):
        # diffusion
        S *= np.exp(drift + vol * Z[:, k])

        # jumps
        jump_mask = N_jump[:, k] > 0
        if np.any(jump_mask):
            Y = np.random.normal(mu_J, sigma_J, size=jump_mask.sum())
            S[jump_mask] *= np.exp(Y)

        sumS += S

    A = sumS / n_steps
    payoff = np.maximum(A - K_real, 0.0)

    price = disc * payoff.mean()
    stderr = disc * payoff.std(ddof=1) / math.sqrt(len(payoff))

    return price, stderr


# ============================================================
# PINN pricing (real scale, K-normalized training)
# ============================================================

@torch.no_grad()
def pinn_price_real(model, S0_real, K_real, r, sigma, t0=0.0):
    S_tilde = torch.tensor([[S0_real / K_real]], device=DEVICE, dtype=DTYPE)
    I_tilde = torch.tensor([[0.0]], device=DEVICE, dtype=DTYPE)
    t = torch.tensor([[t0]], device=DEVICE, dtype=DTYPE)
    rr = torch.tensor([[r]], device=DEVICE, dtype=DTYPE)
    ss = torch.tensor([[sigma]], device=DEVICE, dtype=DTYPE)

    X = torch.cat([S_tilde, I_tilde, t, rr, ss], dim=1)
    V_tilde = model(X).item()

    return K_real * V_tilde


# ============================================================
# Global evaluation (NO region filtering)
# ============================================================

def evaluate_pinn_vs_mc_jump(
    model,
    S_grid,
    K, r, sigma, T,
    lambda_jump, mu_J, sigma_J,
    n_paths_mc=200_000
):
    pinn_prices = []
    mc_prices   = []

    for S in S_grid:
        p_pinn = pinn_price_real(model, S, K, r, sigma)

        p_mc, _ = mc_asian_call_arith_jump(
            S, K, r, sigma,
            lambda_jump, mu_J, sigma_J,
            T=T,
            n_paths=n_paths_mc
        )

        pinn_prices.append(p_pinn)
        mc_prices.append(p_mc)

    pinn_prices = np.array(pinn_prices)
    mc_prices   = np.array(mc_prices)

    mae  = np.mean(np.abs(pinn_prices - mc_prices))
    rmse = np.sqrt(np.mean((pinn_prices - mc_prices)**2))

    return {
        "S_grid": S_grid,
        "PINN": pinn_prices,
        "MC": mc_prices,
        "MAE": mae,
        "RMSE": rmse
    }


In [13]:

S0 = 75.0
K  = 75.0
r  = 0.03
sigma = 0.32
T = 1.0

lambda_jump = 2.188
mu_J        = 0.0196
sigma_J     = 0.1817

# Griglia standard
S_grid_eval = np.linspace(0, S_max_real, 100)

results = evaluate_pinn_vs_mc_jump(
    model,
    S_grid_eval,
    K, r, sigma, T,
    lambda_jump, mu_J, sigma_J
)

print("MAE :", results["MAE"])
print("RMSE:", results["RMSE"])


MAE : 0.03182087216315745
RMSE: 0.04156515159110514
